<a href="https://colab.research.google.com/github/Anthei0774/Advent-of-Code/blob/main/2018/---%20Day%2022%3A%20Mode%20Maze%20---.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
puzzle = '''depth: 510
target: 10,10'''

with open('22.txt') as f: puzzle = f.read()

puzzle = puzzle.split('\n')
puzzle = [ line.split(': ') for line in puzzle ]
puzzle

depth = int(puzzle[0][1])
target = tuple(map(int, puzzle[1][1].split(',')))

################################################################################
# PART 1

cave = {}

def erosion_from_pos(pos):
    global target, depth, cave

    (x, y) = pos
    geo_index = None

    if pos == (0, 0):
        geo_index = 0
    elif pos == target:
        geo_index = 0
    elif y == 0:
        geo_index = x * 16807
    elif x == 0:
        geo_index = y * 48271
    else:
        prev_x = (x - 1, y)
        if prev_x not in cave: create_cave_position(prev_x)
        prev_y = (x, y - 1)
        if prev_y not in cave: create_cave_position(prev_y)
        geo_index = cave[ prev_x ]['erosion'] * cave[ prev_y ]['erosion']
    return (geo_index + depth) % 20183

#
def create_cave_position(pos):
    global cave

    erosion = erosion_from_pos(pos)
    terrain = None
    if erosion % 3 == 0:
        terrain = 'rocky'
    elif erosion % 3 == 1:
        terrain = 'wet'
    else:
        terrain = 'narrow'
    cave[pos] = {'erosion' : erosion, 'terrain' : terrain}


cave = {}
(x, y) = target
for i in range(x + 1):
    for j in range(y + 1):
        create_cave_position( (i, j) )

# risk level
sum( cave[pos]['erosion'] % 3 for pos in cave )

################################################################################
# PART 2



# maps terrain to tools with the terrain is accesible
terrain_to_tools = {
    'narrow' : ['nothing', 'torch'],
    'rocky' : ['gear', 'torch'],
    'wet' : ['gear', 'nothing'],
}

# maps tools to terrains with the terrains are accesible
tool_to_terrains = {
    'gear' : ['rocky', 'wet'],
    'nothing' : ['narrow', 'wet'],
    'torch' : ['narrow', 'rocky'],
}

# dictionary created for switching gear in the same terrain
tool_and_terrain_to_other_tool = {
    ('gear', 'rocky') : 'torch',
    ('gear', 'wet') : 'nothing',
    ('nothing', 'narrow') : 'torch',
    ('nothing', 'wet') : 'gear',
    ('torch', 'narrow') : 'nothing',
    ('torch', 'rocky') : 'gear',
}

def manhattan_distance(pos1, pos2): return abs(pos1[0] - pos2[0]) + abs(pos1[1] - pos2[1])

def euclidean_distance(pos1, pos2): return (pos1[0] - pos2[0]) ** 2 + (pos1[1] - pos2[1]) ** 2

init = ((0, 0), 'torch')

states = {}
states[init] = {'dist' : 0, 'prev' : None }

states_to_check = [ init ]

iter = 0
while True:

    assert states_to_check != []

    current_state = states_to_check.pop(0)
    assert current_state in states

    (pos, tool) = current_state
    assert pos in cave

    terrain = cave[pos]['terrain']
    assert terrain in tool_to_terrains[tool]

    # print('Pos:', pos, '; tool:', tool, '; terrain:', terrain)

    ### NEIGHBOURS
    # create reachable neighbours with current tool
    (x, y) = pos
    neighbours = [ (x - 1, y), (x + 1, y), (x, y - 1), (x, y + 1) ] # all four adjacent positions
    neighbours = [ n for n in neighbours if 0 <= n[0] and 0 <= n[1] ] # not moving down from the map

    for n in neighbours:
        if n not in cave:
            create_cave_position(n)

    neighbours = [ n for n in neighbours if cave[n]['terrain'] in tool_to_terrains[tool] ] # terrain is reachable with current tool
    # print('Reachable neighbours:', neighbours)

    for n in neighbours:
        dist = states[current_state]['dist'] + 1
        new = (n, tool)

        if (new not in states) or ( (new in states) and (dist < states[new]['dist']) ):
            states[new] = {'dist' : dist, 'prev' : current_state}
            states_to_check.append(new)


    # also add switching to other tool
    other_tool = tool_and_terrain_to_other_tool[ (tool, terrain) ]

    dist = states[current_state]['dist'] + 7
    new = (pos, other_tool)

    if (new not in states) or ( (new in states) and (dist < states[new]['dist']) ):
        states[new] = {'dist' : dist, 'prev' : current_state}
        states_to_check.append(new)

    states_to_check = list(set(states_to_check))

    # WROTE A PIECE CR@P CODE, BUT SPITS OUT THE GOOD RESULT AFTER 10 MINS - PROBLEM IS WITH THIS SORTING
    states_to_check = sorted(states_to_check, key = lambda s: manhattan_distance(s[0], target) + states[s]['dist'] )

    d = min(manhattan_distance(s[0], target) for s in states_to_check)
    if iter % 1000 == 0:
        print('Iter:', iter, '; next states to check:', len(states_to_check), '; distance:', d, '\n')

    iter += 1
    # if iter == 100000:
    #     break

    if (target, 'torch') in states:
        print('Target found in', iter, 'iterations')
        break

states[ (target, 'torch') ]['dist']

In [ ]:
def draw():
    global cave, target

    X = max(p[0] for p in cave) + 1
    Y = max(p[1] for p in cave) + 1

    image = ''

    for y in range(Y):
        for x in range(X):
            p = (x, y)

            if p == (0, 0):
                image += 'X'
                continue
            if p == target:
                image += 'T'
                continue

            if p not in cave: create_cave_position(p)

            if cave[p]['terrain'] == 'rocky':
                image += '.'
            elif cave[p]['terrain'] == 'wet':
                image += '='
            else:
                assert cave[p]['terrain'] == 'narrow'
                image += '|'
        image += '\n'

    print(image)


draw()